# Monthly Cross-Sectional Regression (CSR) — Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for `csr_build.ipynb`,
the model's single **monthly** cross-sectional regression. Each month it regresses
realized stock returns on the full USE4 exposure matrix (country + 55 industries +
12 styles) under the cap-weighted industry constraint, and ships two deliverables:
the **factor returns** (the 68 factor-return series) and the **specific
(idiosyncratic) returns** — the per-stock residuals that feed the specific-risk
model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output
as deeply untrustworthy until audited. Follow the stages linearly.

This is the **single canonical source** of factor returns — the one production
CSR in the model. The country build (04) is anchor-only and runs no parallel
regression; this stage owns the full engine — identification, the cap-weighted
industry constraint, the exact reparametrization, and both deliverables.

**Monthly vs daily.** This spec covers the **monthly** production CSR only. The
daily sibling — the same engine run per trading day, feeding the
factor-covariance (06) and specific-risk (07) builds — is specified in the
companion `daily_csr_spec.ipynb`. The reference audit `csr_audit.md` currently
covers the **monthly build only**; the daily CSR is not yet separately audited
(this will be revisited when the audits are brought up to date).

✅ **PDF SPEC (USE4 Methodology Notes §2.1; Empirical Notes §4).** Factor and
specific returns are estimated each period by a single weighted cross-sectional
regression of asset returns on factor exposures; weights are proportional to
√mcap (the idiosyncratic-variance model); the industry factor returns satisfy
the cap-weighted constraint `Σ_j w_jt · f_ind,j,t = 0`, so the country factor
return tracks the cap-weighted market.

**The model.** For month-transition (t, t+1] with n stocks:

```
r = X f + u        X = [1 | D | S]        weights v ∝ √mcap (normalized)
```

- `1` — country column (unit exposure)
- `D` — 55 industry dummies (one 0/1 per stock)
- `S` — 12 standardized style exposures
- `f` — factor returns (the first deliverable)
- `u` — specific returns (the second deliverable), `u = r − X f`

**Identification.** The 55 industry columns sum to the country column, so X′X
is singular; USE4 resolves it with the cap-weighted constraint
`Σ_j w_jt f_ind,j,t = 0`. The country factor's structural role (why an intercept
makes the industries pure relative bets) is explained in
`04_country_factor/country_spec.ipynb`; this spec documents the regression
engine that implements the constraint (§4) and the two deliverables. The
*validation CSR* you built there is this same engine run once as a self-check
under looser sample rules; the production CSR adds a minimum-sample skip
(complete-case n < 100, §4), so it solves ~303 of 329 month-transitions where
the validation CSR solves 327. The lower count is by design.

> **Your task.** The spec and the reference audit (`csr_audit.md`) are provided;
> `csr_build.ipynb` is yours to write. This step runs **right after
> `04_country_factor`** — it consumes the country anchor and every exposure
> deliverable — and directly before its daily sibling:
> `04 country → 05 csr_build → 05 daily_csr_build → 06 fcov_build →
> 07 specific_risk_build → 08 risk_decomp_build`.

## §1. Deliverables and output schema

**Factor returns:** `data/out/csr_factor_returns.parquet`, zstd, statistics=True.

| Column | Type | Description |
|---|---|---|
| `signal_date` | Date | Exposure date t (start of the return month) |
| `ret_date` | Date | End of the return month (the next signal date) |
| `factor` | String | `country`, the 55 industry names, or the 12 style score names |
| `factor_type` | String | `country` / `industry` / `style` (grouping convenience) |
| `f` | Float64 | Factor return for the month (null if the column was degenerate that month) |
| `r2` | Float64 | Weighted cross-sectional R² of the month's regression (repeated per row) |
| `n_stocks` | UInt32 | Stocks in the month's regression (repeated per row) |

68 rows per solved transition (1 + 55 + 12) — 20,604 rows over 303 solved
transitions on the reference run.

**Specific returns:** `data/out/csr_specific_returns.parquet`, zstd, statistics=True.

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | Exposure date t |
| `ret_date` | Date | End of the return month |
| `in_estu` | Bool | True = ESTU regression residual (in-sample); False = non-ESTU coverage stock (out-of-sample, fitted `f` applied) |
| `y` | Float64 | Realized excess log return over (t, t+1] |
| `y_hat` | Float64 | Factor-explained return `X f` |
| `u` | Float64 | Specific (idiosyncratic) return, `u = y − y_hat` |
| `w_reg` | Float64 | √mcap weight, `√mcap_i / Σ_{j∈ESTU realized} √mcap_j` — one scale for both groups (ESTU rows sum to 1 per date) |

Specific returns cover the **whole coverage universe**: every ESTU regression-universe
stock-date actually used in each month's CSR (`in_estu = True`), plus every non-ESTU
coverage stock-date with complete exposures and a realized return (`in_estu = False`),
obtained by applying that transition's fitted `f` out-of-sample. These residuals are the
raw input to the specific-risk model (specific-variance forecasting) and to portfolio
risk decomposition; the specific-risk module forecasts the coverage universe, so it needs
the non-ESTU rows too. (Reference run: 1,097,079 rows — ESTU 637,392 + non-ESTU
459,687 across 10,699 tickers.)

## §2. Pipeline

```
STAGE 0  Setup, parameters
STAGE 1  Anchor: read country_use4.parquet (country exposure + √mcap weights + mcap)
STAGE 2  Monthly returns: compound daily excess log returns over each
         signal-month (stock and benchmark) via the shared daily panel you
         built in 01.5_daily (data/out/daily_returns.parquet)
STAGE 3  Exposure matrix: ESTU (from the anchor) + the non-ESTU coverage matrix
         (industries_use4: in_estu, mcap, industry), each with the 12 style scores
STAGE 4  CSR: per month-transition, constrained WLS → factor returns f and ESTU
         specific returns u = y − X f, then apply f out-of-sample to the non-ESTU
         coverage stocks (same f, no re-fit)
STAGE 5  Ship: csr_factor_returns.parquet + csr_specific_returns.parquet
         (specific returns flagged in_estu)
STAGE 6  Validate
```

**PIT discipline:** exposures dated t explain returns over (t, t+1]; the
regression uses only information known at t.

**Position in the pipeline:** runs immediately **after `04_country_factor`** —
it reads the country anchor and every exposure deliverable (the 12
`<slug>_use4.parquet` style files, `industries_use4.parquet`,
`industry_weights_use4.parquet`) — and **before** its daily sibling
`daily_csr_build.ipynb`, which feeds 06 and 07:

```
04 country → 05 csr_build → 05 daily_csr_build → 06 fcov_build
           → 07 specific_risk_build → 08 risk_decomp_build
```

The anchor is produced by the country build (anchor-only); this stage reads it
and owns the regression — factor returns and the specific-return deliverable.

## §3. STAGE 0 — Parameters

```python
# ✅ PDF SPEC — country exposure is definitional; WLS weights ∝ √mcap;
#    constraint: cap-weighted industry factor returns sum to zero per date.
COUNTRY_EXPOSURE = 1.0

# 💡 DEFAULT (NOT IN PDF) — validation tolerances
FC_CORR_MIN    = 0.99    # corr(f_c, monthly benchmark excess return)
CONSTRAINT_TOL = 1e-10   # max |Σ_j w_j f_ind,j| over dates
R2_MEAN_MIN    = 0.15    # mean weighted cross-sectional R² floor
COND_MAX       = 1e6     # max condition number of the scaled design

# 💡 DEFAULT (NOT IN PDF) — calendar, sample rules, gates
CALENDAR_START   = date(1999, 1, 1)   # monthly rebalance calendar start
MIN_SAMPLE       = 100                # skip transitions below this (returns frame
                                      # or complete-case sample; §4)
UNASSIGNED_LABEL = "UNASSIGNED (fallback ladder)"
UNASSIGNED_MAX   = 50                 # gate on dropped unclassifiable stock-dates

# The 12 style factors: slug → score column in <slug>_use4.parquet
# (the score column name is the `factor` key in csr_factor_returns.parquet)
STYLES = {"size": "size_score", "bp": "btop_score", "dyld": "dyld_score",
          "eyld": "eyld_score", "gro": "gro_score", "lev": "lev_score",
          "liq": "liq_score", "beta": "beta_score", "mom": "mom_score",
          "resvol": "resvol_score", "nlb": "nlb_score", "nls": "nls_score"}
```

**No estimation switches.** The CSR estimates nothing on the exposure side; it
*consumes* the exposure matrix. The pipeline's design stance is **estimate-free**:
missing style scores are never imputed, so the in-ESTU exposure matrix is
intentionally incomplete and the CSR falls back to **complete-case regression**
— any ESTU stock-date missing one of the 12 scores drops from that month's fit,
counted and reported. If you later choose to impute missing exposures, the
engine, weights, and constraint are identical — the only difference is how many
stock-dates the complete-case fallback drops (and how many early transitions
clear the minimum-sample rule, §4).

## §4. STAGE 4 — Constrained WLS + specific returns

Per month-transition (t, t+1] with n stocks: **y** is the excess log return
compounded over the trading days owned by t; **X = [1 | D | S]** is the country
column, 55 industry dummies, and 12 standardized style scores; **v ∝ √mcap** at
t, renormalized over the realized regression sample (the anchor's `w_reg`
rescaled after the complete-case and return drops; assert it still equals
`√mcap / Σ_realized √mcap` to 1e-9 — this ties the anchor scale to the shipped
`w_reg`).

❓ **NOT IN PDF — constraint implementation.** USE4 states the constraint, not
the algorithm.

💡 **DEFAULT — exact reparametrization (restricted least squares).** Eliminate
the **largest-cap-weight present industry e** at each date (numerically safest
divisor): for j ≠ e the factor return is free; `f_e = −(Σ_{j≠e} w_j f_j)/w_e`.
Columns transform as `D̃_j = D_j − (w_j/w_e)·D_e`, the reduced system
`[1 | D̃ | S]` is solved by WLS (√v-scaled `lstsq`), and `f_e` is recovered.
Exact — the choice of e changes nothing but conditioning. Assert the reduced
design has full rank.

❓ **NOT IN PDF — degenerate factor-dates.** gro, beta and nlb carry all-zero
exposure columns on a handful of early-1999 dates (annual-data and benchmark
warm-ups — see the style specs in `02_style_factors`).

💡 **DEFAULT.** Drop a style column from the regression on any date where its
exposure vector is identically zero; record `f = null` for that factor-date —
the null is deliberate, and every downstream consumer must expect it. Assert
surviving style columns are not near-constant (std > 1e-8).

❓ **NOT IN PDF — industries with no members that date.** All 55 industries are
present in the full-ESTU panel at every date (validated upstream), but the
complete-case and return drops can empty an industry on a given early date
(23 industry factor-dates on the reference run).

💡 **DEFAULT.** Exclude absent industries from that date's design and from the
constraint sum; record `f = null` for those factor-dates. Non-ESTU coverage
stocks whose industry was absent from that month's ESTU fit are dropped from
the extension (their `f_ind` is undefined; 39 stock-dates on the reference run).

❓ **NOT IN PDF — return convention.** Monthly horizon is implied; the exact
compounding is not specified.

💡 **DEFAULT.** Sum of daily excess **log** returns over the owned trading days
(the shared trading-day → signal-date ownership mapping: a trading day belongs
to month-end t when its *previous* trading day is ≥ t, so exposures at t
strictly precede every return they explain), consistent with every time-series
factor in this pipeline. Stocks with no trading days in the month (delisted at
t) are dropped from that month's regression and counted.

❓ **NOT IN PDF — unclassifiable stocks.** The industries panel labels stocks
its fallback ladder could not classify (`UNASSIGNED (fallback ladder)`); they
carry no industry factor and no constraint weight.

💡 **DEFAULT.** Drop them from the CSR sample, counted and gated tiny
(`UNASSIGNED_MAX = 50`; 0 ESTU and 28 non-ESTU stock-dates on the reference
run).

❓ **NOT IN PDF — risk-free publication lag.** The Ken French RF series
publishes with a lag, so the daily panel's excess returns are null for the
most recent weeks — whole months at the calendar tail have no usable y.

💡 **DEFAULT.** Skip transitions with no excess-return data (gated ≤ 3, all
within the last three calendar months); picked up on the next data refresh. On
the reference run: 2 skipped (2026-04-30, 2026-05-29 starts).

❓ **NOT IN PDF — thin transitions.** Estimate-free, the earliest months
(1999–2000) have whole style columns uncovered, so the complete-case sample
collapses to a cross-section barely larger than the 68-column design.

💡 **DEFAULT.** Skip any transition whose returns frame or complete-case sample
has fewer than `MIN_SAMPLE = 100` rows. On the reference run 24 early
transitions skip this way; together with the 2 RF-lag skips the production CSR
solves **303 of 329** transitions (sample sizes min 100 / median 2,164 /
max 2,388). This is the deliberate difference from the validation CSR in
`04_country_factor/country_spec.ipynb`, which imposes no minimum-sample rule
and solves 327 — same engine, looser sample rules.

**What this step adds — specific returns.**

✅ **PDF SPEC.** The regression residual *is* the specific return: the part of
an asset's return not explained by its factor exposures. USE4 forecasts
specific risk from these residuals.

For each solved transition, after `f` is recovered:

```
ŷ_i = X_i · f          (factor-explained return)
u_i = y_i − ŷ_i        (specific return)
```

❓ **NOT IN PDF — which universe gets a specific return.** USE4 forecasts
specific risk for the whole coverage universe; the regression only fits ESTU.

💡 **DEFAULT — ship the whole coverage universe, flagged `in_estu`.**
- **ESTU (`in_estu = True`)** — the regression residuals, the exact by-product of
  the fit, where `u ⟂ X` holds by construction.
- **Non-ESTU (`in_estu = False`)** — apply *this* transition's fitted `f`
  out-of-sample: `ŷ_i = f_c + f_ind[ind_i] + S_i · f_style`, `u_i = y_i − ŷ_i`
  (degenerate styles contribute 0). No re-estimation — the same `f`. Kept
  complete-case: all 12 styles finite (drop nulls **and** NaN — raw descriptors
  can yield NaN scores for non-ESTU names), `mcap > 0`, a realized return, and
  an industry that was present in the ESTU fit so `f_ind` is defined;
  `UNASSIGNED` dropped. These residuals are genuinely out-of-sample, so they run
  *wider* than the ESTU residuals (≈ 0.17 vs 0.09 cross-sectional std) and
  carry no orthogonality guarantee.

The `w_reg` weight uses one scale for both groups —
`√mcap_i / Σ_{j∈ESTU realized} √mcap_j` — so ESTU weights sum to 1 per date and the
non-ESTU weights are directly comparable (a consumer can pool the two and
renormalize over the coverage universe).

**Identities the fit guarantees** (asserted in validation):

- `y = ŷ + u` exactly, for every row (definition).
- `Σ_i v_i u_i = 0` per date over **ESTU** — the WLS residual is orthogonal to the
  intercept (the country column of ones), i.e. the cap-weighted ESTU specific
  return is zero. Non-ESTU residuals are out-of-sample and need not satisfy this.

## §5. STAGE 6 — Validation contract

| check | gate |
|---|---|
| 1 | specific-return identity `y = ŷ + u` to machine precision (< 1e-12), all rows |
| 2 | max over dates of \|Σ_i w_i u_i\| < 1e-9 **over ESTU** — residual ⊥ country (cap-weighted ESTU specific return is zero) |
| 3 | corr(f_c, monthly benchmark excess return) > 0.99 — the USE4 signature (they report ≈ 0.999; 0.9982 observed) |
| 4 | max over dates of \|Σ_j w_j f_ind,j\| < 1e-10 — constraint holds to machine precision |
| 5 | max condition number of the √v-scaled design < 1e6 (49.9 observed) |
| 6 | mean weighted cross-sectional R² > 0.15 (0.252 observed) |
| 7 | coverage-flag integrity — no stock-date is both ESTU and non-ESTU; both groups populated |
| 8 | non-ESTU specific-return std finite and sane (0 < std < 1); reported against the ESTU std (OOS ≥ in-sample expected) |

These checks stand on their own — the CSR is the single canonical source, so
there is no external series to reconcile against. The country-factor signature
(check 3, corr(f_c, market) ≈ 0.998) is the structural sanity test the country
build used to carry; it now lives here, with the regression that produces f_c.
Checks 7–8 guard the non-ESTU extension: the orthogonality identity (check 2) is
ESTU-only by construction, so the non-ESTU residuals are validated by coverage
and a finite, wider-than-ESTU spread instead.

**Risk-structure battery (checks 9–15).** Beyond the per-regression identities,
validate the shipped factor-return panel as the *risk structure* the covariance
model (06) will be built on: pivot the monthly country + 12-style returns over
the solved transitions (drop null rows, annualize vols by √12) and gate:

| check | gate |
|---|---|
| 9 | 68 factor rows for every solved transition (min = max = 68) |
| 10 | zero null `f` among country + the 12 styles |
| 11 | the market dominates: country annualized vol strictly above every style's |
| 12 | beta couples to the market: corr(f_beta, f_country) > 0.50 (0.78 observed) |
| 13 | styles near-orthogonal: max \|style–style corr\| < 0.70 (0.48 observed) |
| 14 | conditioning: condition number of the 13×13 factor-return correlation matrix < 50 (19.4 observed) |
| 15 | sane vol levels: country annualized vol ∈ [8, 25]%; all 55 industry annualized vols within [2, 45]% |

Check 10 coexists with §4's degenerate-date rule because the early-1999 dates
that carry all-zero style columns fall inside the thin transitions the
production CSR skips; if a future data vintage ever solves one of those months,
the null is by design — revisit the gate, not the engine.

Plus informational prints: f_c annualized vol (16.1% observed) and monthly
RMSE(f_c − benchmark) (0.28pp observed); the annualized-vol hierarchy (country
and each style, sorted) and each style's correlation with country; the R²
distribution (median 0.236 observed); per-transition drop diagnostics
(return-missing, exposure-incomplete, UNASSIGNED); CSR sample sizes;
eliminated-industry frequency.

## §6. Master list of ❓ NOT-IN-PDF decisions

This stage owns the regression engine, so the full list lives here.

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | Constraint implementation | Exact reparametrization, eliminating the largest-cap-weight present industry per date | Lagrangian / pseudo-inverse on the augmented system | Never — algebraically identical; numerically safest exact form |
| 2 | Eliminated industry | argmax cap weight per date | Fixed industry across dates | Never — exact algebra either way |
| 3 | Return convention | Sum of daily excess log returns over owned trading days; partial months kept | Simple-return compounding; drop partial months | If a downstream consumer standardizes differently |
| 4 | Degenerate factor-dates | Drop the all-zero style column (and any member-less industry) that date, f = null — a deliberate, documented null | Drop the whole date | Never — dropping the date discards 60+ valid factor returns to avoid 1 null |
| 5 | Unclassifiable stocks | Drop `UNASSIGNED (fallback ladder)` rows from the CSR sample, counted and gated (≤ 50) | Assign to a synthetic 56th industry | If the unassigned population ever grows past the gate |
| 6 | RF publication lag | Skip transitions with no excess returns (≤ 3, RF-lag tail only); picked up on the next data refresh | Fall back to raw returns for the tail | Never — mixing raw and excess returns in one series is worse than a short tail |
| 7 | Constraint weights vs CSR sample | Constraint uses full-ESTU cap weights even though ~0.33% of stocks drop for missing returns | Renormalize weights over the realized sample | Never — USE4 defines the weights on the ESTU; the discrepancy is negligible and documented |
| 8 | Specific-return universe | **Whole coverage universe** — ESTU residuals (`in_estu=True`) + non-ESTU coverage stocks via out-of-sample fitted `f` (`in_estu=False`), complete-case | ESTU residuals only | Done — the shipped design. Revisit only if the coverage-universe definition changes |
| 9 | Non-ESTU complete-case | Require all 12 styles finite (drop nulls **and** NaN), `mcap>0`, a realized return, and an ESTU-present industry; UNASSIGNED dropped | Impute missing non-ESTU exposures | Never — imputing exposures we never estimated would fabricate residuals |
| 10 | `w_reg` scale | `√mcap_i / Σ_{j∈ESTU realized} √mcap_j` for both groups — one scale, ESTU sums to 1 per date, non-ESTU comparable | Normalize each group within itself; store raw √mcap | If the specific-risk module wants coverage-universe weights — pool and renormalize |
| 11 | `factor_type` column | Add `country`/`industry`/`style` tag for downstream grouping | Infer from the factor name downstream | Never — costs one column, saves every consumer a lookup table |
| 12 | Specific returns store in_estu, y, ŷ, u, w_reg | Keep all (auditable: `y = ŷ + u` re-checkable; `in_estu` separates in/out-of-sample; `w_reg` for weighted specific variance) | Store `u` only | If storage matters — `u` + `in_estu` suffice for variance |
| 13 | Thin transitions | Skip transitions with returns frame or complete-case sample < 100 (24 early months; 303 of 329 solved vs the 04 validation CSR's 327) | Solve them anyway, as 04's validation CSR does | If you later choose to impute missing exposures, the early months clear the threshold (~327 solve) — re-gate then |